# 01 — Splits stratifiés KB / val / test + anti-fuite

```text
Création et gel des 3 splits (KB / val / test) avec contrôle anti-fuite.

Ordre (important pour garantir la disjonction) :
  1. test  <- COCO val2017   (échantillon stratifié équilibré)
  2. val   <- COCO train2017 (même sampler que test, tuning)
  3. kb    <- COCO train2017 sans val (couverture des 80 classes + distribution naturelle)
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [1]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [2]:
import argparse, math, random
from collections import Counter, defaultdict
from common import (COMPLEXITY_BINS, INDOOR_SUPERCATS, OUTDOOR_SUPERCATS, complexity_label, context_label,
                    env_info, get_logger, load_config, mkdirs, paths, read_json, set_seed, sha256_ids, write_json)
log = get_logger("splits")

### `load_pool()`
Construit un enregistrement par image éligible.

In [3]:
def load_pool(ann_dir, split_name, cfg):
    """Construit un enregistrement par image éligible."""
    caps = read_json(ann_dir / f"captions_{split_name}.json")
    inst = read_json(ann_dir / f"instances_{split_name}.json")
    cat = {c["id"]: c for c in inst["categories"]}
    imgs = {i["id"]: i for i in inst["images"]}
    by_caps = defaultdict(list)
    for a in caps["annotations"]:
        by_caps[a["image_id"]].append((a["id"], a["caption"]))
    by_obj = defaultdict(list)
    for a in inst["annotations"]:
        if a.get("iscrowd", 0) == 0:
            by_obj[a["image_id"]].append(a)
    n_cap = cfg["captions_per_image"]; s = cfg["sampling"]
    pool, reasons = {}, Counter()
    for iid, im in imgs.items():
        if len(by_caps[iid]) < n_cap:
            reasons["moins de 5 captions"] += 1; continue
        if len(by_obj[iid]) < s["min_objects"]:
            reasons["aucune annotation d'objet"] += 1; continue
        if min(im["width"], im["height"]) < s["min_side_px"]:
            reasons["image trop petite"] += 1; continue
        cats = sorted({a["category_id"] for a in by_obj[iid]})
        supers = sorted({cat[c]["supercategory"] for c in cats})
        n_obj = len(by_obj[iid])
        person = 1 in cats
        interaction = (person and len(cats) >= 2) or (sum(a["category_id"] == 1 for a in by_obj[iid]) >= 2) \
            or (len(supers) >= 2 and "animal" in supers)
        pool[iid] = {
            "image_id": iid, "source": split_name, "file_name": im["file_name"], "coco_url": im.get("coco_url", ""),
            "width": im["width"], "height": im["height"], "n_objects": n_obj, "cats": cats, "supercats": supers,
            "complexity": complexity_label(n_obj), "context": context_label(supers), "interaction": bool(interaction),
        }
    log.info("%s: %d images, %d éligibles | exclusions: %s", split_name, len(imgs), len(pool), dict(reasons))
    return pool, {c["id"]: c["name"] for c in cat.values()}, reasons

### `greedy_cover()`
Couverture gloutonne : on traite la classe la plus rare encore sous-couverte, et parmi ses images on prend

In [4]:
def greedy_cover(pool, chosen, cats_all, budget, rng, min_per_cat=1):
    """Couverture gloutonne : on traite la classe la plus rare encore sous-couverte, et parmi ses images on prend
    celle qui couvre le plus d'autres besoins (aléa pour départager). Complexité ~ O(budget x |images de la classe|)."""
    count = Counter()
    for i in chosen:
        count.update(pool[i]["cats"])
    by_cat = defaultdict(list)
    for i in sorted(pool):
        for c in pool[i]["cats"]:
            by_cat[c].append(i)
    taken = set(chosen)
    while len(chosen) < budget:
        need = {c for c in cats_all if count[c] < min_per_cat and any(j not in taken for j in by_cat[c])}
        if not need:
            break
        c0 = min(need, key=lambda c: len(by_cat[c]))
        best, best_sc = [], -1
        for i in by_cat[c0]:
            if i in taken:
                continue
            sc = sum(1 for c in pool[i]["cats"] if c in need and count[c] < min_per_cat)
            if sc > best_sc:
                best, best_sc = [i], sc
            elif sc == best_sc:
                best.append(i)
        i = rng.choice(best)
        chosen.append(i); taken.add(i); count.update(pool[i]["cats"])
    return chosen

### `sample_eval()`
Échantillon d'évaluation stratifié : complexité x contexte, quota d'interactions, couverture des classes.

In [5]:
def sample_eval(pool, n, cfg, rng, exclude=()):
    """Échantillon d'évaluation stratifié : complexité x contexte, quota d'interactions, couverture des classes."""
    s = cfg["sampling"]
    pool = {k: v for k, v in pool.items() if k not in set(exclude)}
    cats_all = sorted({c for v in pool.values() for c in v["cats"]})
    chosen = greedy_cover(pool, [], cats_all, budget=int(0.5 * n), rng=rng, min_per_cat=1)
    # quotas par complexité
    tgt = {k: int(round(v * n)) for k, v in s["eval_complexity_target"].items()}
    while sum(tgt.values()) != n:
        k = max(tgt, key=tgt.get) if sum(tgt.values()) > n else "medium"
        tgt[k] += -1 if sum(tgt.values()) > n else 1
    have = Counter(pool[i]["complexity"] for i in chosen)
    rest = {k: max(0, tgt[k] - have[k]) for k in tgt}
    # si la couverture a déjà dépassé un quota, on re-équilibre sur les autres strates
    deficit = sum(rest.values()) - (n - len(chosen))
    while deficit > 0:
        k = max(rest, key=rest.get); rest[k] -= 1; deficit -= 1
    while sum(rest.values()) < n - len(chosen):
        rest["medium"] += 1
    n_int_target = math.ceil(s["eval_min_interaction_frac"] * n)
    ids_by = defaultdict(list)
    for i in sorted(pool):
        if i not in chosen:
            ids_by[pool[i]["complexity"]].append(i)
    for k in ids_by:
        rng.shuffle(ids_by[k])
    for comp, q in rest.items():
        cand = ids_by[comp]
        # rotation de contexte pour la variété (indoor/outdoor/mixed)
        by_ctx = defaultdict(list)
        for i in cand:
            by_ctx[pool[i]["context"]].append(i)
        order = ["indoor", "outdoor", "mixed"]; picked = []
        while len(picked) < q and any(by_ctx.values()):
            for c in order:
                if by_ctx[c] and len(picked) < q:
                    # priorité aux interactions tant que le quota global n'est pas atteint
                    cur_int = sum(pool[j]["interaction"] for j in chosen + picked)
                    lst = by_ctx[c]
                    pick = next((j for j in lst if pool[j]["interaction"]), lst[0]) if cur_int < n_int_target else lst[0]
                    lst.remove(pick); picked.append(pick)
        chosen += picked
    return chosen[:n]

### `sample_kb()`

In [6]:
def sample_kb(pool, n, cfg, rng, exclude):
    s = cfg["sampling"]
    pool = {k: v for k, v in pool.items() if k not in set(exclude)}
    cats_all = sorted({c for v in pool.values() for c in v["cats"]})
    chosen = greedy_cover(pool, [], cats_all, budget=int(0.6 * n), rng=rng, min_per_cat=s["kb_min_images_per_category"])
    # le reste : tirage aléatoire uniforme -> distribution naturelle COCO (le retrieval doit refléter les données réelles)
    rest = [i for i in sorted(pool) if i not in set(chosen)]
    rng.shuffle(rest)
    chosen += rest[: n - len(chosen)]
    return chosen

### `summarize()`

In [7]:
def summarize(pool, ids, cat_names):
    c = Counter(); ctx = Counter(); inter = 0; cover = Counter()
    for i in ids:
        c[pool[i]["complexity"]] += 1; ctx[pool[i]["context"]] += 1; inter += pool[i]["interaction"]
        cover.update(pool[i]["cats"])
    return {"n": len(ids), "complexity": dict(c), "context": dict(ctx), "interaction_frac": inter / max(1, len(ids)),
            "classes_covered": len(cover), "min_images_per_class": min(cover.values()) if cover else 0,
            "classes_missing": [cat_names[k] for k in cat_names if k not in cover]}

### `main()`

In [8]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exclude", help="JSON d'IDs invalides à exclure (re-tirage déterministe)")
    args = ap.parse_args()
    cfg = load_config(); p = paths(cfg); mkdirs(p); set_seed(cfg["seed"])
    rng = random.Random(cfg["seed"])
    ann = p["raw"] / "annotations"
    bad = set(read_json(p["audit"] / "invalid_images.json").get("ids", [])) if args.exclude else set()

    pool_val2017, cat_names, rs_v = load_pool(ann, "val2017", cfg)
    pool_train2017, _, rs_t = load_pool(ann, "train2017", cfg)
    pool_val2017 = {k: v for k, v in pool_val2017.items() if k not in bad}
    pool_train2017 = {k: v for k, v in pool_train2017.items() if k not in bad}
    sz = cfg["sizes"]

    test_ids = sample_eval(pool_val2017, sz["test"], cfg, random.Random(cfg["seed"] + 1))
    val_ids = sample_eval(pool_train2017, sz["val"], cfg, random.Random(cfg["seed"] + 2))
    kb_ids = sample_kb(pool_train2017, sz["kb"], cfg, random.Random(cfg["seed"] + 3), exclude=val_ids)

    # ---- audit anti-fuite (identifiants)
    assert not set(kb_ids) & set(val_ids), "KB et val se chevauchent"
    assert len(set(kb_ids)) == len(kb_ids) and len(set(val_ids)) == len(val_ids) and len(set(test_ids)) == len(test_ids)
    # train2017 et val2017 ont des espaces d'IDs disjoints, on le vérifie quand même
    assert not (set(pool_train2017) & set(pool_val2017)), "IDs communs train2017/val2017 ?!"
    assert not set(test_ids) & (set(kb_ids) | set(val_ids)), "fuite test -> kb/val"

    splits = {"kb": (kb_ids, pool_train2017), "val": (val_ids, pool_train2017), "test": (test_ids, pool_val2017)}
    meta = {"version": cfg["release_version"], "seed": cfg["seed"], "sizes": {k: len(v[0]) for k, v in splits.items()},
            "env": env_info(), "exclusion_reasons": {"train2017": dict(rs_t), "val2017": dict(rs_v)},
            "id_hash_sha256": {}, "summary": {}, "invalid_excluded": sorted(bad)}
    for name, (ids, pool) in splits.items():
        ids = sorted(ids)
        recs = [{k: pool[i][k] for k in ("image_id", "source", "file_name", "coco_url", "width", "height",
                                          "n_objects", "complexity", "context", "interaction")} for i in ids]
        write_json(p["splits"] / f"{name}.json", {"split": name, "version": cfg["release_version"], "images": recs})
        meta["id_hash_sha256"][name] = sha256_ids(ids)
        meta["summary"][name] = summarize(pool, ids, cat_names)
        log.info("%-4s n=%d | %s", name, len(ids), meta["summary"][name])
    write_json(p["splits"] / "splits_meta.json", meta)
    log.info("splits écrits -> %s", p["splits"])

## Lancer l'étape

In [9]:
import sys
sys.argv = ["notebook"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

11:33:22 | INFO | val2017: 5000 images, 4940 éligibles | exclusions: {"aucune annotation d'objet": 48, 'image trop petite': 12}
11:33:59 | INFO | train2017: 118287 images, 117041 éligibles | exclusions: {"aucune annotation d'objet": 1021, 'image trop petite': 225}
11:34:34 | INFO | kb   n=10000 | {'n': 10000, 'complexity': {'simple': 2646, 'crowded': 3076, 'complex': 1716, 'medium': 2562}, 'context': {'indoor': 3741, 'outdoor': 3773, 'mixed': 2486}, 'interaction_frac': 0.615, 'classes_covered': 80, 'min_images_per_class': 66, 'classes_missing': []}
11:34:34 | INFO | val  n=100 | {'n': 100, 'complexity': {'simple': 25, 'medium': 30, 'complex': 30, 'crowded': 15}, 'context': {'outdoor': 35, 'indoor': 35, 'mixed': 30}, 'interaction_frac': 0.69, 'classes_covered': 80, 'min_images_per_class': 1, 'classes_missing': []}
11:34:34 | INFO | test n=100 | {'n': 100, 'complexity': {'simple': 25, 'complex': 30, 'crowded': 15, 'medium': 30}, 'context': {'mixed': 30, 'outdoor': 33, 'indoor': 37}, 'int

## Résultats

In [10]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Résumé des splits

In [11]:
meta = json.load(open(P["splits"] / "splits_meta.json"))
display(pd.DataFrame({s: {"n": v["n"], "classes couvertes": v["classes_covered"], "min img/classe": v["min_images_per_class"],
                           "% interaction": round(100 * v["interaction_frac"], 1)} for s, v in meta["summary"].items()}))
display(pd.DataFrame({s: v["complexity"] for s, v in meta["summary"].items()}).fillna(0).astype(int).rename_axis("complexité"))
display(pd.DataFrame({s: v["context"] for s, v in meta["summary"].items()}).fillna(0).astype(int).rename_axis("contexte"))
print("hash SHA-256 des listes d'IDs :"); print(json.dumps(meta["id_hash_sha256"], indent=1))
print("exclusions éligibilité :", meta["exclusion_reasons"])

,kb,val,test
n,10000.0,100.0,100.0
classes couvertes,80.0,80.0,80.0
min img/classe,66.0,1.0,1.0
% interaction,61.5,69.0,61.0


,kb,val,test
complexité,,,
simple,2646,25,25
crowded,3076,15,15
complex,1716,30,30
medium,2562,30,30


,kb,val,test
contexte,,,
indoor,3741,35,37
outdoor,3773,35,33
mixed,2486,30,30


hash SHA-256 des listes d'IDs :
{
 "kb": "75dd463be67a5cb7d6c70259ad4faee52a14c5b3be87c747d0b43471c436c914",
 "val": "8c9487885ca36cad18ccd189c99f60c74afe48c6e4df7f00e9a5446e232b00d2",
 "test": "4b7682064003c0925b62f4d9f345f50ac297da6f79e43d3f6865c620cd1754f2"
}
exclusions éligibilité : {'train2017': {"aucune annotation d'objet": 1021, 'image trop petite': 225}, 'val2017': {"aucune annotation d'objet": 48, 'image trop petite': 12}}


### Aperçu du split test

In [12]:
display(pd.DataFrame(json.load(open(P["splits"] / "test.json"))["images"]).head(10))

,image_id,source,file_name,coco_url,width,height,n_objects,complexity,context,interaction
0,2261,val2017,000000002261.jpg,http://images.cocodataset.org/val2017/00000000...,640,427,2,simple,mixed,True
1,14473,val2017,000000014473.jpg,http://images.cocodataset.org/val2017/00000001...,640,427,8,complex,outdoor,True
2,16228,val2017,000000016228.jpg,http://images.cocodataset.org/val2017/00000001...,640,440,20,crowded,outdoor,True
3,23781,val2017,000000023781.jpg,http://images.cocodataset.org/val2017/00000002...,640,427,7,complex,mixed,False
4,23937,val2017,000000023937.jpg,http://images.cocodataset.org/val2017/00000002...,640,428,6,complex,outdoor,False
5,43816,val2017,000000043816.jpg,http://images.cocodataset.org/val2017/00000004...,640,426,5,medium,mixed,True
6,44877,val2017,000000044877.jpg,http://images.cocodataset.org/val2017/00000004...,640,427,6,complex,mixed,True
7,47585,val2017,000000047585.jpg,http://images.cocodataset.org/val2017/00000004...,424,640,7,complex,mixed,True
8,54654,val2017,000000054654.jpg,http://images.cocodataset.org/val2017/00000005...,407,640,2,simple,indoor,True
9,55528,val2017,000000055528.jpg,http://images.cocodataset.org/val2017/00000005...,640,480,7,complex,indoor,True
